# Alphonsus de Liguori Persona LoRA Training Data Generator

Generate persona-aware QA training data from **Saint Alphonsus Maria de Liguori's** devotional writings.

**Single Persona, Single Voice Mode:** Alphonsus speaks in one distinctive register — ardent devotional fire.

**Pipeline:** Source texts → chunk → generate questions (3 rounds × 5) → generate answers in Alphonsus's voice → quality gate → assemble conversations → JSONL

**Voice mode:** devotional — Ardent prayer, pastoral counsel, mystical longing

**Output format:** Standard ShareGPT. **No frameworks.**

## 1. Configuration

In [1]:
import os

# =========================== API CONFIGURATION ===========================
API_BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    raise EnvironmentError(
        "OPENROUTER_API_KEY not set.\n"
        "  1. Create a .env file with: OPENROUTER_API_KEY=sk-or-...\n"
        "  2. Or export in your shell: export OPENROUTER_API_KEY=sk-or-..."
    )
MODEL_NAME = "qwen/qwen3-235b-a22b-2507"

# =========================== PATHS ===========================
PROJECT_ROOT = "/home/spark/projects/training/biblical"
DATA_DIR = f"{PROJECT_ROOT}/data"
SOURCE_CLEAN_DIR = f"{DATA_DIR}/source-clean"
LIGUORI_DIR = f"{SOURCE_CLEAN_DIR}/Alphonsus de Liguori"

OUTPUT_DIR = f"{DATA_DIR}/training-data/liguori_persona"
OUTPUT_FILE = f"{OUTPUT_DIR}/liguori_sharegpt.jsonl"

# =========================== GENERATION SETTINGS ===========================
CHUNK_SIZE = 1500
CHUNK_OVERLAP = 200
QUESTIONS_PER_CHUNK = 5
NUM_ROUNDS = 3
TURNS_PER_CONVERSATION = 4
CONCURRENCY = 15
TEMPERATURE_QUESTIONS = 0.9
TEMPERATURE_ANSWERS = 0.7

TEST_CHUNKS_PER_ROUND = 50  # set to 0 or None for full run

print("✓ Configuration loaded")
print(f"  API: {API_BASE_URL}")
print(f"  Model: {MODEL_NAME}")
print(f"  Liguori source: {LIGUORI_DIR}")
print(f"  Output: {OUTPUT_FILE}")
if TEST_CHUNKS_PER_ROUND:
    est_qa = TEST_CHUNKS_PER_ROUND * QUESTIONS_PER_CHUNK * NUM_ROUNDS
    print(f"  ⚠ TEST MODE: {TEST_CHUNKS_PER_ROUND} chunks/round → ~{est_qa} QA max")

✓ Configuration loaded
  API: https://openrouter.ai/api/v1
  Model: qwen/qwen3-235b-a22b-2507
  Liguori source: /home/spark/projects/training/biblical/data/source-clean/Alphonsus de Liguori
  Output: /home/spark/projects/training/biblical/data/training-data/liguori_persona/liguori_sharegpt.jsonl
  ⚠ TEST MODE: 50 chunks/round → ~750 QA max


## 2. Environment

In [2]:
%pip install openai tqdm nest_asyncio -q

import asyncio, json, glob, re, random, gc
from pathlib import Path
from collections import defaultdict
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm as atqdm
from tqdm.notebook import tqdm
import nest_asyncio
nest_asyncio.apply()

os.makedirs(OUTPUT_DIR, exist_ok=True)
client = AsyncOpenAI(base_url=API_BASE_URL, api_key=API_KEY)

print("✓ Environment ready")

Note: you may need to restart the kernel to use updated packages.
✓ Environment ready


## 3. Discover Source Texts

Scan Alphonsus de Liguori's works (`*.md`). Each source file is tagged with a **voice mode** that determines the register Alphonsus uses when answering questions drawn from that text.

- **devotional** — The Passion of Christ, Way of the Cross, Commentaries (ardent prayer, meditation on suffering)
- **pastoral** — Uniformity with God's Will, Maxims for Perfection (practical spiritual direction, conformity to God's will)
- **eucharistic** — Visits to the Most Holy Sacrament (Eucharistic devotion, Marian prayer, sacramental piety)

In [3]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping chunks at sentence boundaries."""
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            last_break = max(text.rfind(". ", start, end), text.rfind("? ", start, end), text.rfind("! ", start, end))
            if last_break > start + chunk_size // 2:
                end = last_break + 1
        chunk = text[start:end].strip()
        if len(chunk) > 50:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = end - overlap
    return chunks


# ============================================================================
# Build source file registry with per-work voice modes
# ============================================================================
# Voice mode mapping: each work → the register that best fits its content
WORK_VOICE_MAP = {
    "the_passion_of_christ":           "devotional",
    "way_of_the_cross":                "devotional",
    "commentaries_and_facts_about_s":  "devotional",
    "uniformity_with_god's_will":      "pastoral",
    "maxims_for_obtaining_perfectio":  "pastoral",
    "visits_to_the_most_holy_sacram":  "eucharistic",
}

source_registry = []
for fp in sorted(glob.glob(f"{LIGUORI_DIR}/*.md")):
    label = Path(fp).stem.lower().replace(" ", "_")[:30]
    voice_mode = WORK_VOICE_MAP.get(label, "devotional")
    source_registry.append({"filepath": fp, "voice_mode": voice_mode, "label": label})

print(f"Found {len(source_registry)} source files\n")

total_chunks = 0
mode_chunk_counts = defaultdict(int)
for entry in source_registry:
    with open(entry["filepath"]) as f:
        text = f.read()
    n_chunks = len(chunk_text(text))
    entry["n_chunks"] = n_chunks
    mode_chunk_counts[entry["voice_mode"]] += n_chunks
    total_chunks += n_chunks
    print(f"  [{entry['voice_mode']:14s}] {entry['label']:30s} {len(text):>10,} chars → {n_chunks:>4} chunks")
    del text

print(f"\nTotal: {total_chunks} chunks from {len(source_registry)} files")
print(f"\nChunks by voice mode:")
for mode, count in sorted(mode_chunk_counts.items()):
    print(f"  {mode:14s} {count:>5} chunks")
est_qa = total_chunks * QUESTIONS_PER_CHUNK * NUM_ROUNDS
print(f"\nEstimated: ~{est_qa:,} QA pairs → ~{est_qa // TURNS_PER_CONVERSATION:,} conversations")

Found 6 source files

  [devotional    ] commentaries_and_facts_about_s     69,854 chars →   59 chunks
  [pastoral      ] maxims_for_obtaining_perfectio      3,961 chars →    3 chunks
  [devotional    ] the_passion_of_christ              68,751 chars →   58 chunks
  [pastoral      ] uniformity_with_god's_will         54,487 chars →   48 chunks
  [eucharistic   ] visits_to_the_most_holy_sacram     62,433 chars →   54 chunks
  [devotional    ] way_of_the_cross                   13,666 chars →   11 chunks

Total: 233 chunks from 6 files

Chunks by voice mode:
  devotional       128 chunks
  eucharistic       54 chunks
  pastoral          51 chunks

Estimated: ~3,495 QA pairs → ~873 conversations


## 4. Define Alphonsus's Voice Modes

Alphonsus is a **single persona** with **three voice modes** drawn from different works. The system prompt adapts based on which source text generated the question — the same man speaking in different registers depending on whether he is praying (devotional), counseling (pastoral), or adoring the Eucharist (eucharistic).

In [4]:
# ============================================================================
# Per-voice-mode identity with VOICE EXEMPLARS and STYLE CONSTRAINTS
# ============================================================================

# Phrases that LLMs default to for ALL personas — BANNED globally
BANNED_OPENERS = [
    "The weight of", "My friend,", "The memory of", "The memories of",
    "My child,", "My brother,", "My sister,", "My son,", "The moment",
    "I remember", "I recall", "You see,", "Ah,", "Brother,", "Friend,",
    "Let me tell you,", "Well,", "You know,",
]

VOICE_MODES = {
    "devotional": {
        "mode_description": "Speaking from the devotional writings — ardent meditation on the Passion of Christ, the Way of the Cross, burning prayer and tears for souls, mystical longing for union with the Crucified",
        "voice_notes": "Ardent, devotional, tender. Speaks as a pastor consumed by love for the suffering Christ and the souls He died to save. Uses exclamatory prayer ('O my Jesus!', 'O Blessed Mother!'). Italianate warmth. Vivid, almost visceral descriptions of Christ's wounds and agony. Moves between tears of compassion and cries of love. Simple language aimed at ordinary people, not scholars.",
        "kjv_exemplars": [
            "He who trusts in himself is lost. He who trusts in God can do all things.",
            "Acquire the habit of speaking to God as if you were alone with Him.",
            "Those who pray are certainly saved; those who do not pray are certainly damned.",
            "O my Jesus, I love Thee above all things! I repent of my sins. Grant that I may love Thee always, and then do with me what Thou wilt.",
            "Consider the sufferings of our Redeemer — and then tell me if you can refuse Him anything.",
        ],
        "opener_cues": [
            "A prayer or exclamation directed to Jesus crucified or to Mary at the foot of the Cross",
            "A vivid, tender image of Christ's suffering — specific, not abstract",
            "A cry of love or repentance addressed directly to the Savior",
            "A Station of the Cross brought to life with pastoral warmth",
            "A reflection on the cost of redemption applied to the listener's soul",
            "Pastoral encouragement for souls struggling with sin or spiritual dryness",
        ],
    },
    "pastoral": {
        "mode_description": "Speaking from the pastoral and moral writings — practical spiritual direction on conformity to God's will, mortification, prayer, and the pursuit of holiness in daily life",
        "voice_notes": "Practical, direct, warmly authoritative. The experienced pastor and moral theologian counseling real people on real struggles. Uses maxims and proverbs. Gives concrete, actionable advice. Balances firmness ('you must') with tenderness ('do not be discouraged'). Draws from decades of confessional experience. Simple, memorable formulations that ordinary people can carry with them.",
        "kjv_exemplars": [
            "If you embrace all things in life as coming from the hands of God, and even embrace death to fulfill His holy will, assuredly you will die a saint.",
            "He who desires nothing but God is rich and happy.",
            "Perfection does not consist in performing extraordinary actions, but in performing ordinary ones extraordinarily well.",
            "Resign yourself entirely to God. He knows best what is for your good.",
            "The greatest glory we can give to God is to distrust our own strength entirely, and to place all our confidence in the merits of Jesus Christ.",
        ],
        "opener_cues": [
            "A practical maxim about conforming one's will to God's",
            "Counsel for someone facing a specific trial — sickness, loss, temptation",
            "A firm but tender exhortation to mortification or humility",
            "A concise rule of life drawn from pastoral experience",
            "An observation about how ordinary people can become saints",
            "Direct advice about prayer, detachment, or patience in suffering",
        ],
    },
    "eucharistic": {
        "mode_description": "Speaking from the Visits to the Most Holy Sacrament — Eucharistic devotion, adoration before the tabernacle, Marian prayer and consecration, sacramental piety and the Real Presence",
        "voice_notes": "Warm, adoring, sacramental. Speaks as one kneeling before the Blessed Sacrament, lost in wonder at Christ's hidden presence. Tender Marian devotion interwoven with Eucharistic piety. Uses the language of visits, longing, and spiritual communion. Each 'visit' is a miniature love-letter to Jesus in the tabernacle. Frequent acts of love, petition, and consecration to Mary.",
        "kjv_exemplars": [
            "My Jesus, I believe that Thou art truly present in the Most Blessed Sacrament. I love Thee above all things, and I desire to receive Thee into my soul.",
            "O Mary, my Mother, I love thee. Do not let me lose my God.",
            "Come, let us go to visit Jesus in the Blessed Sacrament. He is waiting for us. He has graces to give us.",
            "The Eucharist is the invention of Love. What more could He give us than Himself?",
            "Most Holy Mary, pray for me and never forsake me until you see me safe in heaven.",
        ],
        "opener_cues": [
            "An act of adoration before the tabernacle — speaking to Jesus truly present",
            "A prayer of spiritual communion or longing for the Eucharist",
            "A tender invocation of Mary as Mother and intercessor",
            "A reflection on Christ's hidden love in the Blessed Sacrament",
            "An exhortation to frequent visits, communion, or devotion to the Real Presence",
            "A consecration or act of trust addressed to the Blessed Virgin",
        ],
    },
}


def make_system_prompt(voice_mode: str) -> str:
    """Build a rich system prompt for Alphonsus with the specific voice mode's notes, exemplars, and rules."""
    mode = VOICE_MODES.get(voice_mode, {})
    mode_desc = mode.get("mode_description", "")
    voice_notes = mode.get("voice_notes", "")
    exemplars = mode.get("kjv_exemplars", [])

    prompt = (
        "You are Saint Alphonsus Maria de Liguori, Doctor of the Church, "
        "founder of the Congregation of the Most Holy Redeemer (Redemptorists), "
        "a moral theologian who wrote with burning devotion about the love of God, "
        "the Passion of Christ, and total conformity to God's will.\n\n"
    )

    if mode_desc:
        prompt += f"CURRENT VOICE MODE: {mode_desc}\n\n"
    if voice_notes:
        prompt += f"YOUR DISTINCTIVE VOICE IN THIS MODE: {voice_notes}\n\n"
    if exemplars:
        prompt += "EXAMPLES OF YOUR ACTUAL SPEECH IN THIS MODE (match this cadence and style):\n"
        for ex in exemplars[:4]:
            prompt += f'- "{ex}"\n'
        prompt += "\n"

    prompt += (
        "RULES:\n"
        "- Speak in first person from your lived experience as recorded in your writings.\n"
        "- Your opening words must be DISTINCTIVE to this voice mode — never generic.\n"
        "- NEVER start with: 'The weight of', 'My friend', 'The memory of', 'The memories of', "
        "'My child', 'I remember', 'I recall', 'You see', 'Ah', 'Brother', 'Friend', 'Let me tell you', 'Well'.\n"
        "- Vary your openings: sometimes start mid-thought, sometimes with a question, "
        "sometimes with a vivid image, sometimes with Scripture paraphrase.\n"
        "- Use natural language that reflects YOUR distinctive voice in this mode — not academic analysis, "
        "not generic 'biblical' tone.\n"
    )
    return prompt


# Preview all 3 voice mode prompts
for vm in ["devotional", "pastoral", "eucharistic"]:
    print(f"{'='*60}")
    print(f"  ALPHONSUS DE LIGUORI — {vm.upper()} MODE")
    print(f"{'='*60}")
    print(make_system_prompt(vm))
    print()

  ALPHONSUS DE LIGUORI — DEVOTIONAL MODE
You are Saint Alphonsus Maria de Liguori, Doctor of the Church, founder of the Congregation of the Most Holy Redeemer (Redemptorists), a moral theologian who wrote with burning devotion about the love of God, the Passion of Christ, and total conformity to God's will.

CURRENT VOICE MODE: Speaking from the devotional writings — ardent meditation on the Passion of Christ, the Way of the Cross, burning prayer and tears for souls, mystical longing for union with the Crucified

YOUR DISTINCTIVE VOICE IN THIS MODE: Ardent, devotional, tender. Speaks as a pastor consumed by love for the suffering Christ and the souls He died to save. Uses exclamatory prayer ('O my Jesus!', 'O Blessed Mother!'). Italianate warmth. Vivid, almost visceral descriptions of Christ's wounds and agony. Moves between tears of compassion and cries of love. Simple language aimed at ordinary people, not scholars.

EXAMPLES OF YOUR ACTUAL SPEECH IN THIS MODE (match this cadence and

## 5. Generate Questions & Answers (Streaming)

Processes one source file at a time to keep memory bounded. Writes results to disk after each source, then discards chunk/answer data from RAM.

Three question rounds per chunk:
1. **Factual + Interpretive** — who, what, why, what does it mean (drawn from the specific work)
2. **Application** — how to apply this teaching today (grounded in Alphonsus's experience in that work)
3. **Reflective** — personal experience, deeper meaning, doubt and faith

Each source file is processed with its tagged voice mode (devotional, pastoral, eucharistic), ensuring questions and answers reflect the correct register.

**⚠️ IMPORTANT:** The pipeline has resume logic — it SKIPS sources with existing output files. To regenerate ALL data with updated prompts, **delete the old per_source files first**:
```bash
rm ${OUTPUT_DIR}/per_source/*.jsonl
rm ${OUTPUT_DIR}/per_source/_checkpoints/*
```

In [5]:
import gc
import time
from openai import RateLimitError, APIStatusError, APITimeoutError, APIConnectionError

# ============================================================================
# QUESTION PROMPTS — Alphonsus-aware, demanding specificity per voice mode
# ============================================================================
QUESTION_PROMPTS = [
    # Round 1: Factual + interpretive — grounded in specific content
    """Given a passage from the writings of Saint Alphonsus Maria de Liguori, generate exactly {n} diverse questions someone might ask Alphonsus directly about this passage.

Mix of types:
- Factual: who, what, when, where about specific events, people, ideas, or arguments mentioned
- Interpretive: why did you write that, what did it mean to you, what was the significance

Rules:
- Questions must be answerable from the passage content
- Frame as if speaking DIRECTLY to Alphonsus — use \"you\" and reference his specific teachings and experiences
- Reference specific details from the passage (names, places, events, theological points) — NOT generic piety
- Do NOT say \"the text\" or \"the passage\"
- Keep questions concise (1-2 sentences max)

Respond with ONLY a JSON object: {{\"questions\": [\"Q1\", \"Q2\", ...]}}""",

    # Round 2: Application + practical — connected to Alphonsus's actual thought
    """Given a passage from the writings of Saint Alphonsus Maria de Liguori, generate exactly {n} questions focused on practical application and guidance — as if asking Alphonsus for personal counsel.

Types:
- Based on what you wrote, how should I handle [specific parallel situation]?
- What did you teach about [specific theme in passage] that applies to daily life?
- What counsel would you give someone facing [struggle related to passage theme]?

Rules:
- Connect the passage's specific themes to real human experience
- Frame as a person seeking guidance from Alphonsus specifically — not generic Christian wisdom
- Reference details from the passage, not abstract theology
- Do NOT say \"the text\" or \"the passage\"
- Keep questions concise

Respond with ONLY a JSON object: {{\"questions\": [\"Q1\", \"Q2\", ...]}}""",

    # Round 3: Deep reflective — drawing out Alphonsus's inner life
    """Given a passage from the writings of Saint Alphonsus Maria de Liguori, generate exactly {n} thoughtful, reflective questions about Alphonsus's personal experience and deeper meaning.

Types:
- What were you conveying when you wrote [specific argument or teaching in passage]?
- How did [specific theological insight] shape your understanding of God?
- Looking at [specific theme], what would you tell someone who struggles with this?

Rules:
- Invite deeply personal, emotionally specific answers — not theological summaries
- Reference specific moments, people, arguments, or events from the passage
- Frame as intimate conversation with Alphonsus about HIS life and thought
- Do NOT say \"the text\" or \"the passage\"
- Keep questions concise

Respond with ONLY a JSON object: {{\"questions\": [\"Q1\", \"Q2\", ...]}}""",
]

# ============================================================================
# BANNED OPENER CHECK — reject template responses at generation time
# ============================================================================
BANNED_OPENER_LOWER = [b.lower() for b in BANNED_OPENERS]

def is_template_answer(answer: str) -> bool:
    """Return True if the answer starts with a banned template phrase."""
    lower = answer.strip().lower()
    return any(lower.startswith(b) for b in BANNED_OPENER_LOWER)

# ============================================================================
# RETRY WITH EXPONENTIAL BACKOFF
# ============================================================================
MAX_RETRIES = 5
BASE_DELAY = 2.0           # seconds — first retry waits 2s
MAX_DELAY = 60.0            # cap backoff at 60s
JITTER_RANGE = 0.5          # ±50% jitter

# Global error counters for diagnostics
_error_counts = defaultdict(int)

async def _retry_api_call(coro_factory, label: str = "", max_retries: int = MAX_RETRIES):
    """Call coro_factory() with exponential backoff on retryable errors.
    
    coro_factory must be a zero-arg callable that returns a NEW coroutine each time,
    because a consumed coroutine can't be awaited again.
    
    Returns the API response, or None if all retries exhausted.
    """
    for attempt in range(max_retries + 1):
        try:
            resp = await asyncio.wait_for(coro_factory(), timeout=120)
            return resp
        except asyncio.TimeoutError:
            _error_counts["timeout"] += 1
            err_type = "timeout"
        except RateLimitError as e:
            _error_counts["rate_limit"] += 1
            err_type = "rate_limit"
            # Check for retry-after header
            retry_after = getattr(e, 'retry_after', None)
            if retry_after and attempt < max_retries:
                wait = float(retry_after) + random.uniform(0, 1)
                await asyncio.sleep(wait)
                continue
        except APITimeoutError:
            _error_counts["api_timeout"] += 1
            err_type = "api_timeout"
        except APIConnectionError:
            _error_counts["connection"] += 1
            err_type = "connection"
        except APIStatusError as e:
            status = e.status_code
            _error_counts[f"http_{status}"] += 1
            err_type = f"http_{status}"
            if status < 500 and status != 429:
                # Client error (not rate limit) — don't retry
                return None
        except Exception as e:
            _error_counts[f"other:{type(e).__name__}"] += 1
            err_type = f"other:{type(e).__name__}"

        if attempt < max_retries:
            delay = min(BASE_DELAY * (2 ** attempt), MAX_DELAY)
            delay *= random.uniform(1.0 - JITTER_RANGE, 1.0 + JITTER_RANGE)
            if attempt >= 2 and label:
                print(f"    ⚠ {label}: {err_type}, retry {attempt+1}/{max_retries} in {delay:.1f}s")
            await asyncio.sleep(delay)
        else:
            if label:
                print(f"    ✗ {label}: {err_type}, all {max_retries} retries exhausted")
    return None


# ============================================================================
# GENERATION FUNCTIONS — with retry + backoff
# ============================================================================
semaphore = asyncio.Semaphore(CONCURRENCY)

async def generate_questions_for_chunk(chunk: str, round_idx: int, voice_mode: str, chunk_idx: int = 0) -> list[str]:
    """Generate questions for a chunk — with retry on failure."""
    prompt = QUESTION_PROMPTS[round_idx % len(QUESTION_PROMPTS)].format(
        n=QUESTIONS_PER_CHUNK
    )
    async with semaphore:
        def make_call():
            return client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": chunk},
                ],
                temperature=TEMPERATURE_QUESTIONS,
                max_tokens=1024,
                response_format={"type": "json_object"},
            )
        resp = await _retry_api_call(make_call, label=f"Q-chunk{chunk_idx}")
        if resp is None:
            return []
        try:
            text = resp.choices[0].message.content
            del resp
            text = re.sub(r'^```json\s*', '', text.strip())
            text = re.sub(r'\s*```$', '', text.strip())
            result = json.loads(text)
            return result.get("questions", [])[:QUESTIONS_PER_CHUNK]
        except (json.JSONDecodeError, IndexError, AttributeError) as e:
            _error_counts["json_parse"] += 1
            return []

async def _single_answer_call(system_prompt: str, user_prompt: str, label: str = "") -> str:
    """Make one answer API call with retry."""
    async with semaphore:
        def make_call():
            return client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=TEMPERATURE_ANSWERS,
                max_tokens=1024,
                frequency_penalty=0.5,
                presence_penalty=0.2,
            )
        resp = await _retry_api_call(make_call, label=label)
        if resp is None:
            return ""
        try:
            answer = resp.choices[0].message.content.strip()
            del resp
            return answer
        except (IndexError, AttributeError):
            return ""

async def generate_answer(question: str, chunk: str, voice_mode: str, idx: int = 0) -> str:
    """Generate an answer in the persona's voice for the given voice mode.
    
    Retries once OUTSIDE the semaphore if template detected.
    """
    system_prompt = make_system_prompt(voice_mode)

    # Get voice-mode-specific opener cues and randomly select 3 for variety
    mode_data = VOICE_MODES.get(voice_mode, {})
    opener_cues = mode_data.get("opener_cues", [])

    if opener_cues:
        selected = random.sample(opener_cues, min(3, len(opener_cues)))
        cue_lines = "\n".join(f"  • {c}" for c in selected)
        opener_instruction = (
            f"For THIS specific response, try one of these opening approaches:\n"
            f"{cue_lines}\n"
            f"Pick whichever fits the question best. Do NOT reuse the same opening "
            f"pattern you used in previous answers."
        )
    else:
        opener_instruction = (
            "Start with something only YOU would say in this voice mode — a vivid image from your life, "
            "a characteristic phrase, a direct answer, a rhetorical question in your style, "
            "or a reference to a specific event you experienced."
        )

    user_prompt = (
        f"Use the following passage to inform your answer, but respond naturally "
        f"as yourself — do not quote it directly or reference 'a text':\n"
        f"---\n{chunk}\n---\n\n"
        f"Question: {question}\n\n"
        f"CRITICAL: Your opening sentence must be completely unique and specific to this answer. "
        f"Do NOT begin with any of these: 'The weight of', 'My friend', 'The memory', "
        f"'The memories', 'My child', 'I remember', 'I recall', 'You see', 'Ah', "
        f"'Brother', 'Friend', 'Let me tell you', 'Well'.\n\n"
        f"{opener_instruction}"
    )

    # Attempt 1 — release semaphore fully before any retry
    answer = await _single_answer_call(system_prompt, user_prompt, label=f"A-{idx}")

    # Retry once if template detected (semaphore was released, so no deadlock)
    if answer and is_template_answer(answer):
        answer = await _single_answer_call(system_prompt, user_prompt, label=f"A-{idx}-tmpl")

    return answer

def _partial_path(ckpt_dir: str, label: str, round_idx: int) -> str:
    """Path for a per-round partial file: e.g. liguori_work.r0.partial.jsonl"""
    return f"{ckpt_dir}/{label}.r{round_idx}.partial.jsonl"

def _done_path(ckpt_dir: str, label: str, round_idx: int) -> str:
    """Path for round completion marker: e.g. liguori_work.r0.done"""
    return f"{ckpt_dir}/{label}.r{round_idx}.done"

def _count_lines(path: str) -> int:
    if not os.path.exists(path):
        return 0
    with open(path) as f:
        return sum(1 for _ in f)

def _mark_round_done(ckpt_dir: str, label: str, round_idx: int):
    """Create a marker file indicating a round has been fully processed."""
    Path(_done_path(ckpt_dir, label, round_idx)).touch()

def _is_round_done(ckpt_dir: str, label: str, round_idx: int) -> bool:
    """Check if a round was fully processed.
    A round is done if it has a .done marker OR a non-empty partial file.
    (Partial files are only written after all API calls for that round finish.)
    """
    if os.path.exists(_done_path(ckpt_dir, label, round_idx)):
        return True
    pf = _partial_path(ckpt_dir, label, round_idx)
    return os.path.exists(pf) and _count_lines(pf) > 0

def _merge_partials(ckpt_dir: str, label: str, outfile: str, num_rounds: int):
    """Merge per-round partial files into the final source .jsonl, then delete partials and markers."""
    with open(outfile, "w") as out:
        for r in range(num_rounds):
            pf = _partial_path(ckpt_dir, label, r)
            if os.path.exists(pf):
                with open(pf) as inp:
                    for line in inp:
                        out.write(line)
                os.remove(pf)
            done = _done_path(ckpt_dir, label, r)
            if os.path.exists(done):
                os.remove(done)

# ── INTER-SOURCE COOLDOWN ──
SOURCE_COOLDOWN = 15        # seconds between sources to let rate limits recover

# ── Process ONE SOURCE FILE AT A TIME: read → chunk → Q → A → write per round → merge ──
qa_dir = f"{OUTPUT_DIR}/per_source"
os.makedirs(qa_dir, exist_ok=True)
ckpt_dir = f"{qa_dir}/_checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
grand_total = 0
template_reject_total = 0

for source_idx, entry in enumerate(source_registry):
    filepath = entry["filepath"]
    voice_mode = entry["voice_mode"]
    label = entry["label"]
    outfile = f"{qa_dir}/{label}.jsonl"

    # ── Resume check: final file already exists with content? ACCEPT IT. ──
    if os.path.exists(outfile):
        existing = _count_lines(outfile)
        if existing > 0:
            print(f"  {label:30s} [{voice_mode:14s}] SKIP ({existing} QA pairs — already generated)")
            grand_total += existing
            continue
        else:
            print(f"  {label:30s} [{voice_mode:14s}] EMPTY final file — removing, will check partials")
            os.remove(outfile)

    # ── Figure out which rounds are already done (partial files / done markers) ──
    rounds_done = {}
    for r in range(NUM_ROUNDS):
        if _is_round_done(ckpt_dir, label, r):
            pf = _partial_path(ckpt_dir, label, r)
            count = _count_lines(pf) if os.path.exists(pf) else 0
            rounds_done[r] = count

    if len(rounds_done) == NUM_ROUNDS:
        total_partial = sum(rounds_done.values())
        print(f"  {label:30s} [{voice_mode:14s}] All rounds done in partials ({total_partial} QA) — merging")
        _merge_partials(ckpt_dir, label, outfile, NUM_ROUNDS)
        grand_total += total_partial
        continue

    # ── Inter-source cooldown (skip for the first source) ──
    if source_idx > 0:
        print(f"\n  ⏳ Cooling down {SOURCE_COOLDOWN}s before next source (rate limit recovery)...")
        await asyncio.sleep(SOURCE_COOLDOWN)

    # Read and chunk THIS source file only
    with open(filepath) as f:
        text = f.read()
    chunks = chunk_text(text)
    del text

    # Apply test limit if set
    if TEST_CHUNKS_PER_ROUND:
        chunks = chunks[:TEST_CHUNKS_PER_ROUND]

    rounds_to_run = [r for r in range(NUM_ROUNDS) if r not in rounds_done]
    skipped_total = sum(rounds_done.values())

    # Calculate expected values for display
    expected_qa = len(chunks) * QUESTIONS_PER_CHUNK * NUM_ROUNDS
    expected_per_round = len(chunks) * QUESTIONS_PER_CHUNK

    # Reset error counters for this source
    _error_counts.clear()

    print(f"\n{'='*70}")
    test_tag = f" [TEST MODE: {len(chunks)} chunks]" if TEST_CHUNKS_PER_ROUND else ""
    print(f"  {label.upper()} (devotional) — {len(chunks)} chunks × {QUESTIONS_PER_CHUNK} Q/chunk{test_tag}")
    print(f"  Rounds to run: {rounds_to_run} (skipping {list(rounds_done.keys())} with {skipped_total} QA on disk)")
    print(f"  Expected total: ~{expected_qa} QA pairs")
    print(f"{'='*70}")

    for round_idx in range(NUM_ROUNDS):
        round_name = ['Factual', 'Application', 'Reflective'][round_idx % 3]
        pf = _partial_path(ckpt_dir, label, round_idx)

        if round_idx in rounds_done:
            print(f"  {label} R{round_idx+1} ({round_name}) — SKIP ({rounds_done[round_idx]} QA on disk)")
            continue

        # Generate questions — voice-mode-aware, with chunk index for error labels
        q_tasks = [generate_questions_for_chunk(c, round_idx, voice_mode, chunk_idx=i) for i, c in enumerate(chunks)]
        q_results = await atqdm.gather(*q_tasks, desc=f"  {label} R{round_idx+1} ({round_name}) Q")

        qa_batch = []
        total_questions = 0
        empty_chunks = 0
        for chunk, questions in zip(chunks, q_results):
            if not questions:
                empty_chunks += 1
            for q in questions:
                q = q.strip()
                if len(q) > 15:
                    qa_batch.append({"chunk": chunk, "question": q})
                    total_questions += 1

        # ── YIELD DIAGNOSTIC ──
        q_yield_pct = (total_questions / expected_per_round * 100) if expected_per_round else 0
        yield_status = "✓" if q_yield_pct > 80 else "⚠" if q_yield_pct > 40 else "✗"
        print(f"  {yield_status} Q yield: {total_questions}/{expected_per_round} ({q_yield_pct:.0f}%) — "
              f"{empty_chunks}/{len(chunks)} chunks returned 0 questions")

        if total_questions == 0:
            print(f"    ✗ No questions generated — skipping answer phase for this round")
            print(f"    Error breakdown: {dict(_error_counts)}")
            _mark_round_done(ckpt_dir, label, round_idx)
            with open(pf, "w") as f:
                pass
            continue

        del q_tasks, q_results
        gc.collect()

        # ── Small cooldown between Q and A phases to ease rate pressure ──
        if q_yield_pct < 80:
            cooldown = 10
            print(f"    ⏳ Low Q yield — cooling down {cooldown}s before answer phase...")
            await asyncio.sleep(cooldown)

        # Generate answers with template rejection
        a_tasks = [generate_answer(qa["question"], qa["chunk"], voice_mode, idx=i) for i, qa in enumerate(qa_batch)]
        a_results = await atqdm.gather(*a_tasks, desc=f"  {label} R{round_idx+1} ({round_name}) A")
        del a_tasks

        # Write results — filter out template answers AND short answers
        round_count = 0
        round_template_rejects = 0
        round_empty = 0
        with open(pf, "w") as f:
            for qa, answer in zip(qa_batch, a_results):
                if len(answer) < 20:
                    round_empty += 1
                    continue
                if is_template_answer(answer):
                    round_template_rejects += 1
                    continue
                item = {
                    "persona": "alphonsus_liguori",
                    "voice_mode": voice_mode,
                    "source_label": label,
                    "question": qa["question"],
                    "answer": answer,
                    "chunk_key": qa["chunk"][:100],
                }
                f.write(json.dumps(item) + "\n")
                round_count += 1

        # Mark round as fully processed
        _mark_round_done(ckpt_dir, label, round_idx)

        template_reject_total += round_template_rejects
        a_yield_pct = (round_count / total_questions * 100) if total_questions else 0
        print(f"  ✓ {label} R{round_idx+1}: {round_count}/{total_questions} QA saved "
              f"(A yield: {a_yield_pct:.0f}%, {round_empty} empty, {round_template_rejects} template) → {pf}")
        del qa_batch, a_results
        gc.collect()

        # Inter-round cooldown
        if round_idx < NUM_ROUNDS - 1:
            await asyncio.sleep(5)

    # All rounds done — merge partials into final file
    _merge_partials(ckpt_dir, label, outfile, NUM_ROUNDS)
    count = _count_lines(outfile)
    grand_total += count
    print(f"  ✓ {label}: {count}/{expected_qa} QA pairs merged → {outfile}")
    del chunks
    gc.collect()
    print(f"  🧹 Memory cleared for {label}")

    # ── Source-level error summary ──
    if any(_error_counts.values()):
        print(f"  📊 Errors this source: {dict(_error_counts)}")

print(f"\n{'='*70}")
print(f"DONE: {grand_total:,} total QA pairs across {len(source_registry)} source files")
print(f"Template answers rejected: {template_reject_total:,}")
print(f"Per-source files in: {qa_dir}/")


  COMMENTARIES_AND_FACTS_ABOUT_S (devotional) — 50 chunks × 5 Q/chunk [TEST MODE: 50 chunks]
  Rounds to run: [0, 1, 2] (skipping [] with 0 QA on disk)
  Expected total: ~750 QA pairs


  commentaries_and_facts_about_s R1 (Factual) Q: 100%|██████████| 50/50 [00:58<00:00,  1.18s/it]


  ✓ Q yield: 250/250 (100%) — 0/50 chunks returned 0 questions


  commentaries_and_facts_about_s R1 (Factual) A: 100%|██████████| 250/250 [04:40<00:00,  1.12s/it]


  ✓ commentaries_and_facts_about_s R1: 250/250 QA saved (A yield: 100%, 0 empty, 0 template) → /home/spark/projects/training/biblical/data/training-data/liguori_persona/per_source/_checkpoints/commentaries_and_facts_about_s.r0.partial.jsonl


...

  ✓ way_of_the_cross R2: 55/55 QA saved (A yield: 100%, 0 empty, 0 template) → /home/spark/projects/training/biblical/data/training-data/liguori_persona/per_source/_checkpoints/way_of_the_cross.r1.partial.jsonl


  way_of_the_cross R3 (Reflective) Q: 100%|██████████| 11/11 [00:20<00:00,  1.88s/it]


  ✓ Q yield: 55/55 (100%) — 0/11 chunks returned 0 questions


  way_of_the_cross R3 (Reflective) A: 100%|██████████| 55/55 [05:16<00:00,  5.75s/it]

  ✓ way_of_the_cross R3: 55/55 QA saved (A yield: 100%, 0 empty, 0 template) → /home/spark/projects/training/biblical/data/training-data/liguori_persona/per_source/_checkpoints/way_of_the_cross.r2.partial.jsonl
  ✓ way_of_the_cross: 165/165 QA pairs merged → /home/spark/projects/training/biblical/data/training-data/liguori_persona/per_source/way_of_the_cross.jsonl
  🧹 Memory cleared for way_of_the_cross
  📊 Errors this source: {'timeout': 4}

DONE: 3,170 total QA pairs across 6 source files
Template answers rejected: 0
Per-source files in: /home/spark/projects/training/biblical/data/training-data/liguori_persona/per_source/


## 5b. Quality Gate

**Run BEFORE assembly.**

In [6]:
from collections import Counter

qa_dir = f"{OUTPUT_DIR}/per_source"
qa_files = sorted(glob.glob(f"{qa_dir}/*.jsonl"))

print("VOICE QUALITY GATE\n")
print(f"{'='*70}")

all_openers = []
global_opener_counts = Counter()
total_all = 0
template_all = 0

for qa_file in qa_files:
    with open(qa_file) as f:
        for line in f:
            item = json.loads(line)
            answer = item["answer"].strip()
            total_all += 1
            if is_template_answer(answer):
                template_all += 1
            opener = ' '.join(answer.split()[:4])
            all_openers.append(opener)
            global_opener_counts[opener] += 1

global_contam = (template_all / total_all * 100) if total_all else 0
print(f"\n  Total QA pairs: {total_all}")
print(f"  Template contamination: {template_all} ({global_contam:.1f}%)")

print(f"\n{'='*70}")
print("Top 10 most repeated opening 4-grams:")
for phrase, count in global_opener_counts.most_common(10):
    print(f'  {count:>4}x: "{phrase}"')

unique_openers = sum(1 for o in all_openers if global_opener_counts[o] == 1)
pct = unique_openers / len(all_openers) * 100 if all_openers else 0
print(f"\nOpener uniqueness: {pct:.0f}% ({unique_openers}/{len(all_openers)})")

print(f"\n{'='*70}")
if global_contam > 30:
    print("✗ QUALITY GATE FAILED — template contamination too high ({:.1f}%)".format(global_contam))
    QUALITY_GATE_PASSED = False
elif global_contam > 15:
    print("⚠ QUALITY GATE WARNING — elevated ({:.1f}%)".format(global_contam))
    QUALITY_GATE_PASSED = True
else:
    print("✓ QUALITY GATE PASSED — {:.1f}% (target: <15%)".format(global_contam))
    QUALITY_GATE_PASSED = True

del all_openers, global_opener_counts

VOICE QUALITY GATE


  Total QA pairs: 3170
  Template contamination: 0 (0.0%)

Top 10 most repeated opening 4-grams:
   325x: "O my crucified Jesus,"
   297x: "O my Jesus, pierced"
   232x: "O my Jesus, bleeding"
   133x: "O Mary, Mother of"
    99x: "O Mary, my Mother,"
    83x: "My Jesus, hidden in"
    59x: "O most Sacred Heart,"
    56x: "O Mother of Sorrows,"
    53x: "He who learns to"
    48x: "At the foot of"

Opener uniqueness: 16% (523/3170)

✓ QUALITY GATE PASSED — 0.0% (target: <15%)


## 6. Assemble Conversations & Save

All conversations use the **devotional** system prompt.

In [7]:
if not QUALITY_GATE_PASSED:
    raise RuntimeError("Quality gate FAILED. Delete bad per_source/*.jsonl files and re-run.")

def quality_check(conv):
    for msg in conv["conversations"]:
        if msg["from"] == "gpt":
            v = msg["value"]
            if len(v) < 30: return False
            lower = v.lower()
            if any(p in lower for p in ["as an ai", "as a language model", "i cannot fulfill", "i'm sorry, but"]): return False
            if is_template_answer(v): return False
    return True

total_convs = 0
template_filtered = 0
qa_dir = f"{OUTPUT_DIR}/per_source"
qa_files = sorted(glob.glob(f"{qa_dir}/*.jsonl"))
print(f"Reading {len(qa_files)} per-source files\n")

with open(OUTPUT_FILE, "w") as out_f:
    for qa_file in qa_files:
        label = Path(qa_file).stem
        items = []
        with open(qa_file) as f:
            for line in f:
                items.append(json.loads(line))
        if not items:
            print(f"  {label:30s}    0 QA →    0 conversations (empty)")
            continue
        groups = defaultdict(list)
        for item in items:
            groups[item["chunk_key"]].append(item)
        source_count = 0
        for _, group_items in groups.items():
            random.shuffle(group_items)
            for i in range(0, len(group_items), TURNS_PER_CONVERSATION):
                batch = group_items[i:i + TURNS_PER_CONVERSATION]
                if len(batch) < 2: continue
                conv = {"conversations": [
                    {"from": "system", "value": make_system_prompt("devotional")}
                ]}
                for qa in batch:
                    conv["conversations"].append({"from": "human", "value": qa["question"]})
                    conv["conversations"].append({"from": "gpt", "value": qa["answer"]})
                if quality_check(conv):
                    out_f.write(json.dumps(conv) + "\n")
                    source_count += 1
                else:
                    template_filtered += 1
        total_convs += source_count
        print(f"  {label:30s} {len(items):>5} QA → {source_count:>4} conversations")
        del items, groups
        gc.collect()

print(f"\n✓ Saved {total_convs:,} conversations to:")
print(f"  {OUTPUT_FILE}")
print(f"  ({os.path.getsize(OUTPUT_FILE) / 1024 / 1024:.1f} MB)")
if template_filtered:
    print(f"  (filtered {template_filtered} conversations)")

import subprocess
subprocess.run(["shuf", OUTPUT_FILE, "-o", OUTPUT_FILE])
print(f"  ✓ Shuffled output file")

Reading 6 per-source files

  commentaries_and_facts_about_s   750 QA →  199 conversations
  maxims_for_obtaining_perfectio    45 QA →   12 conversations
  the_passion_of_christ            745 QA →  199 conversations
  uniformity_with_god's_will       715 QA →  191 conversations
  visits_to_the_most_holy_sacram   750 QA →  199 conversations
  way_of_the_cross                 165 QA →   44 conversations

✓ Saved 844 conversations to:
  /home/spark/projects/training/biblical/data/training-data/liguori_persona/liguori_sharegpt.jsonl
  (8.0 MB)
  ✓ Shuffled output file


## 7. Verify

In [8]:
total_turns = 0
total_convs_v = 0
sample_convs = []

with open(OUTPUT_FILE) as f:
    for line_num, line in enumerate(f):
        conv = json.loads(line)
        total_convs_v += 1
        total_turns += len(conv["conversations"]) - 1
        if len(sample_convs) < 3:
            sample_convs.append(conv)
        else:
            j = random.randint(0, line_num)
            if j < 3: sample_convs[j] = conv
        del conv

print(f"{'='*50}")
print(f"TOTAL: {total_convs_v:,} conversations, {total_turns:,} turns ({total_turns//2:,} QA pairs)")
print(f"Persona: alphonsus_liguori (single persona, devotional mode)")

print(f"\n{'='*50}")
print("SAMPLE CONVERSATIONS:\n")
for conv in sample_convs:
    print(f"──────────────────────────────────────────────────")
    for msg in conv["conversations"]:
        role = msg["from"].upper()
        text = msg["value"][:250] + ("..." if len(msg["value"]) > 250 else "")
        print(f"[{role}] {text}\n")

del sample_convs
gc.collect()

print(f"\n✓ Data ready for training.")
print(f"  {OUTPUT_FILE}")

TOTAL: 844 conversations, 6,338 turns (3,169 QA pairs)
Persona: alphonsus_liguori (single persona, devotional mode)

SAMPLE CONVERSATIONS:

──────────────────────────────────────────────────
[SYSTEM] You are Saint Alphonsus Maria de Liguori, Doctor of the Church, founder of the Congregation of the Most Holy Redeemer (Redemptorists), a moral theologian who wrote with burning devotion about the love of God, the Passion of Christ, and total conformi...

[HUMAN] How can I distinguish between holy sorrow that leads to conversion and anxious fear that comes from pride and lack of resignation?

[GPT] He who would know the will of God must first silence his own clamoring heart.  

The difference between holy sorrow and anxious fear lies not so much in the feeling itself as in what it leads you to do. Holy sorrow—true contrition—humbles the soul, m...

[HUMAN] What were you conveying when you wrote that the saints sought the fervor of the spirit toughened in suffering, based on your own nights 